In [1]:
# Import essential Python packages
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse
from adjustText import adjust_text

# Scientific and statistical packages
import scipy.stats as ss
from skbio import DistanceMatrix
from skbio.stats.ordination import OrdinationResults
from skbio.stats.distance import permanova

# BIOM and Qiime2 related packages
import biom
from biom import load_table
from qiime2 import Artifact

# Gemelli RPCA and rarefaction utilities
from gemelli.rpca import rpca # Optional: import auto_rpca if you use auto_rpca later
from gemelli.utils import qc_rarefaction


In [12]:
biom_tbl = biom.load_table('../Data/16S/Tables/16S_V1-V3_Genus_collapsed.biom')
df = pd.DataFrame(biom_tbl.matrix_data.toarray(), 
                  index=biom_tbl.ids(axis='observation'), 
                  columns=biom_tbl.ids(axis='sample'))

# Display the DataFrame
#print(df)
column_sums = df.sum(axis=0)
#print("Column sums to check for relative abundance:\n", column_sums)

# Check if any column sums are larger than 1 and print True or False
any_larger_than_one = (column_sums > 1.0).any()
print(any_larger_than_one)


# Read the metadata file
metadata_file = '../Metadata/metadata_final_18062024.tsv'
metadata_df = pd.read_csv(metadata_file, sep='\t')
#print(metadata_df)

False


In [13]:
# Ensure SampleID is the index in the metadata DataFrame
metadata_df.set_index('SampleID', inplace=True)

# Merge the abundance DataFrame with the metadata DataFrame on SampleID
merged_df = df.T.merge(metadata_df, left_index=True, right_index=True)

# Check the merged DataFrame
print(merged_df.head())

# Extract the abundance of Cutibacterium and local lesion severity score
cutibacterium_abundance = merged_df['g__Cutibacterium']
local_lesion_severity = merged_df['local_lesion_severity']

# Calculate the correlation
correlation, p_value = pearsonr(cutibacterium_abundance, local_lesion_severity)
print(f"Correlation between Cutibacterium abundance and local lesion severity score: {correlation:.2f} (p-value: {p_value:.3e})")

# Plotting the correlation
plt.figure(figsize=(8, 6))
sns.scatterplot(x=local_lesion_severity, y=cutibacterium_abundance)
plt.title('Correlation between Cutibacterium Abundance and Local Lesion Severity Score')
plt.xlabel('Local Lesion Severity Score')
plt.ylabel('Abundance of Cutibacterium')
plt.axhline(0, color='red', linestyle='--')  # Optional: Add a horizontal line at y=0
plt.axvline(0, color='red', linestyle='--')  # Optional: Add a vertical line at x=0
plt.grid()
plt.tight_layout()
plt.show()

                    g__Acinetobacter   g__Actinomyces   g__Alloprevotella  \
LAMI.RD001.D0.C1                 0.0         0.000000            0.004776   
LAMI.RD001.D0.C2                 0.0         0.000265            0.007164   
LAMI.RD001.D14.C1                0.0         0.000796            0.007960   
LAMI.RD001.D14.C2                0.0         0.002123            0.026532   
LAMI.RD001.D28.C1                0.0         0.000265            0.010082   

                    g__Allorhizobium-Neorhizobium-Pararhizobium-Rhizobium  \
LAMI.RD001.D0.C1                                                 0.0        
LAMI.RD001.D0.C2                                                 0.0        
LAMI.RD001.D14.C1                                                0.0        
LAMI.RD001.D14.C2                                                0.0        
LAMI.RD001.D28.C1                                                0.0        

                    g__Anaerococcus   g__Auricoccus-Abyssicoccus  \
LAMI.R

KeyError: 'g__Cutibacterium'